In [1]:
import pandas as pd

df = pd.read_csv("bike_sharing_hourly.csv")
(df['casual'] + df['registered'] == df['cnt']).mean()


np.float64(1.0)

In [9]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_features = [
    'hr', 'workingday', 'temp', 'hum', 'windspeed',
    'casual', 'registered'
]

X_pca = df[pca_features]

scaler = StandardScaler()
X_pca_scaled = scaler.fit_transform(X_pca)

pca = PCA(n_components=2)
Z = pca.fit_transform(X_pca_scaled)

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [9]:


df["hr_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hr"] / 24)
df["mnth_sin"] = np.sin(2 * np.pi * df["mnth"] / 12)
df["mnth_cos"] = np.cos(2 * np.pi * df["mnth"] / 12)

features = [
    'season',
    'mnth_sin',
    'mnth_cos',
    'weathersit',
    'temp',
    'atemp',
    'hum',
    'windspeed'
]

X = df[features]

y = df['high_demand']

In [10]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

y_binary = (y == 'High').astype(int)

split_index = int(0.75 * len(df))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]
tscv = TimeSeriesSplit(n_splits=5)

In [11]:
elastic_interaction_pipe = Pipeline([
    ('interactions', PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        max_iter=10000,
        random_state=0
    ))
])

In [12]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9]
}

In [14]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_interaction = GridSearchCV(
    elastic_interaction_pipe,
    param_grid,
    cv=tscv,
    scoring='accuracy'
)

grid_interaction.fit(X_train, y_train)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('interactions',
                                        PolynomialFeatures(include_bias=False,
                                                           interaction_only=True)),
                                       ('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=10000,
                                                           penalty='elasticnet',
                                                           random_state=0,
                                                           solver='saga'))]),
             param_grid={'model__C': [0.01, 0.1, 1, 10, 100],
                         'model__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9]},
             scoring='accuracy')

In [17]:
grid_interaction.best_params_

{'model__C': 0.01, 'model__l1_ratio': 0.1}

In [19]:
y_pred = grid_interaction.predict(X_test)
y_prob = grid_interaction.predict_proba(X_test)[:, 1]

In [20]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.7424626006904488

In [21]:
confusion_matrix(y_test, y_pred)

array([[1838,  878],
       [ 241, 1388]])

In [22]:
auc = roc_auc_score(y_test, y_prob)
auc

np.float64(0.8380389814219626)

In [23]:
results = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC AUC'],
    'Value': [accuracy, auc]
})

results

,Metric,Value
0,Accuracy,0.742463
1,ROC AUC,0.838039


In [27]:
interaction_names = grid_interaction.best_estimator_.named_steps['interactions'].get_feature_names_out(features)
best_model = grid_interaction.best_estimator_.named_steps['model']

coef_table = pd.DataFrame({
    'feature': interaction_names,
    'coefficient': best_model.coef_[0]
})

coef_table['abs_coefficient'] = coef_table['coefficient'].abs()

coef_table.sort_values('abs_coefficient', ascending=False).head(20)

,feature,coefficient,abs_coefficient
4,temp,-0.567063,0.567063
13,season hum,0.500875,0.500875
6,hum,0.423540,0.423540
9,season mnth_cos,-0.401418,0.401418
5,atemp,-0.354815,0.354815
30,temp atemp,-0.293152,0.293152
15,mnth_sin mnth_cos,0.258555,0.258555
19,mnth_sin hum,-0.240434,0.240434
11,season temp,-0.225628,0.225628
25,mnth_cos windspeed,0.196816,0.196816


In [42]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest AUC:", rf_auc)

Random Forest Accuracy: 0.706789413118527
Random Forest AUC: 0.7810608485196969


In [27]:
rf_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

rf_importance

,feature,importance
5,hum,0.263653
6,windspeed,0.196452
4,atemp,0.167806
3,temp,0.162874
1,mnth,0.108729
2,weathersit,0.051293
0,season,0.049193


In [28]:
comparison = pd.DataFrame({
    'Model': ['Elastic Net Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy, rf_accuracy],
    'ROC AUC': [auc, rf_auc]
})

comparison

,Model,Accuracy,ROC AUC
0,Elastic Net Logistic Regression,0.739010,0.807066
1,Random Forest,0.761105,0.836322
